# Lesson 01 - AI एजंट्स परिचय

**AI Agents for Beginners** कोर्समधील पहिल्या धड्यात तुमचे स्वागत आहे!

एक **AI एजंट** हा असा प्रोग्राम आहे जो मोठ्या भाषा मॉडेल (LLM) ला त्याच्या तर्कशक्ती इंजिन म्हणून वापरतो आणि वास्तविक जगात *क्रिया* करू शकतो — API कॉल करणे, डेटाबेस क्वेरी करणे, किंवा कोड चालवणे — वापरकर्त्यासाठी एखादे ध्येय साध्य करण्यासाठी.

या नोटबुकमध्ये तुम्ही तुमचा पहिला एजंट तयार कराल: एक **ट्रॅव्हल एजंट** जो सुट्टीसाठी स्थाने सुचवतो. याबरोबरच तुम्हाला शिकायला मिळेल:

1. **Microsoft Agent Framework** वापरून Azure AI Foundry Agent Service शी कनेक्ट करणे.
2. एजंटला एक **टूल** देणे — एक साधी Python फंक्शन जी एजंट कॉल करू शकतो.
3. एजंट चालवणे आणि त्याचा प्रतिसाद तपासणे.
4. एजंटचा प्रतिसाद टोकन-दर-टोकन स्ट्रीम करणे.


## सेटअप

हा नोटबुक चालवण्यापूर्वी, खात्री करा की तुमच्याकडे आहे:

1. **एक Azure AI Foundry प्रकल्प** ज्यात एक चालू चॅट मॉडेल (उदा. `gpt-4o-mini`) आहे.
2. **Azure CLI मध्ये लॉगिन केलेले आहे** — तुमच्या टर्मिनलमध्ये `az login` चालवा.
3. **आवश्यक पर्यावरण चरोंची सेटिंग केली आहे:**
   - `AZURE_AI_PROJECT_ENDPOINT` — तुमच्या Azure AI Foundry प्रकल्पाचा एंडपॉईंट.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — तुमच्या डिप्लॉय केलेल्या मॉडेलचे नाव.

खालील सेलमध्ये तुम्हाला आवश्यक असलेले Python पॅकेजेस इन्स्टॉल होतील.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## आपल्या पहिल्या एजंटची निर्मिती

एजंटला दोन गोष्टींची गरज असते:

- **सूचना** ज्या त्याला सांगतात *तो कोण आहे* आणि *कसा वागत आहे* (एक सिस्टम प्रॉम्प्ट).
- **साधने** — Python फंक्शन्स जे `@tool` ने सजवलेले असतात आणि ज्यांना एजंट माहिती मिळवण्यासाठी किंवा क्रिया करण्यासाठी कॉल करू शकतो.

खालोखाल आम्ही एक सोपी साधन परिभाषित करतो जी लोकप्रिय सुट्टी ठिकाणांची यादी परत करते. वापरकर्ता प्रवास शिफारसी मागितल्यावर एजंट ही साधन वापरेल.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## स्ट्रीमिंग प्रतिसाद

अधिक संवादात्मक अनुभवासाठी तुम्ही एजंटचा प्रतिसाद **स्ट्रीम** करू शकता. पूर्ण उत्तरासाठी प्रतीक्षा करण्याऐवजी, एजंट तयार करत असलेल्या मजकूराचे तुकडे थेट देतो. हे विशेषतः चॅट इंटरफेसमध्ये उपयुक्त आहे जिथे तुम्हाला आउटपुट रिअल टाइममध्ये दाखवायचे असते.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## सारांश

या धड्यात तुम्ही कसे शिकाल:

- **प्रदाता तयार करा** जो `AzureAIProjectAgentProvider` द्वारे Azure AI Foundry Agent Service शी कनेक्ट होतो.
- **टूल परिभाषित करा** `@tool` डेकोरेटर वापरून जेणेकरून एजंट तुमच्या Python फंक्शन्स कॉल करू शकतील.
- **एजंट चालवा** वापरकर्त्याच्या संदेशासह आणि त्याची प्रतिक्रिया प्रिंट करा.
- **प्रतिक्रिया स्ट्रीम करा** वास्तविक वेळी आउटपुटसाठी.

पुढील धड्यात आपण एजंटिक फ्रेमवर्क्स अधिक सखोलपणे पाहू आणि एजंट्सना अधिक शक्तिशाली टूल्स आणि बहु-चरणीय तर्कशक्ती क्षमता कशा द्यायच्या ते शिकू.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
हा दस्तऐवज AI भाषांतर सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) चा वापर करून अनुवादित केला आहे. जरी आम्ही अचूकतेसाठी प्रयत्न करतो, तरी कृपया लक्षात घ्या की स्वयंचलित भाषांतरांमध्ये त्रुटी किंवा अचूकतेची कमतरता असू शकते. मूळ दस्तऐवज त्याच्या मूळ भाषेत अधिकृत स्रोत मानला पाहिजे. महत्त्वाची माहिती असल्यास, व्यावसायिक मानवी भाषांतराची शिफारस केली जाते. या भाषांतराच्या वापरामुळे उद्भवणाऱ्या कोणत्याही गैरसमज किंवा चुकीच्या अर्थलावणीसाठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
